이 노트북은 최종 제출용으로 정리한 버전이다. 시행착오용 DWT/SVM/MLP 코드, 실패 traceback, 중복 학습/복구 셀은 제거하고 최종 실험 파이프라인만 남겼다.

**실행 흐름**  
데이터 로드 → Trial 단위 Train/Test 분할 → 60 Hz Notch → 20~499 Hz Band-pass → 300 ms Window / 50% overlap → Min-Max → Morlet CWT(scale 1~32) → 2D CNN / ResNet18 / DenseNet161 → 평가 → Confusion Matrix → 5-Fold → 계산 비용 비교

> 필요한 경우 `RUN_FULL_EXPERIMENT`와 `RUN_CV`를 `True`로 설정하여 전체 실험을 실행할 수 있다.


## 1. 환경 설정


In [ ]:
!pip -q install PyWavelets scipy scikit-learn matplotlib pandas

import os, glob, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pywt
import torch
import torch.nn as nn

from scipy.signal import iirnotch, butter, filtfilt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from torch.utils.data import TensorDataset, DataLoader
from torchvision.models import resnet18, densenet161

FS = 1000
WIN = 300
HOP = 150
SCALES = np.arange(1, 33)
BATCH_SIZE = 16
EPOCHS = 45
LR = 1e-3
RANDOM_STATE = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. 데이터 다운로드 및 확인

공개 저장소의 A~E 사용자별 CSV 250개를 사용한다. 각 trial은 3초(3000 samples), 2채널이다.


In [ ]:
if not os.path.exists('palm-sEMG-doorknob-filtered'):
    !git clone -q https://github.com/sea3551/palm-sEMG-doorknob-filtered.git

DATA_ROOT = 'palm-sEMG-doorknob-filtered/data'
files = sorted(glob.glob(f'{DATA_ROOT}/**/*.csv', recursive=True))
labels = np.array([os.path.basename(os.path.dirname(f)) for f in files])

print('전체 trial:', len(files))
print(pd.Series(labels).value_counts().sort_index())

sample = np.loadtxt(files[0], delimiter=',', skiprows=1)
print('sample shape:', sample.shape)


## 3. Trial 단위 Train/Test 분할

Window를 만들기 전에 원본 trial 파일을 8:2로 분할한다. 같은 trial에서 생성된 유사 window가 Train/Test 양쪽에 들어가는 데이터 누수를 방지하기 위한 절차다.


In [ ]:
train_files, test_files, train_labels, test_labels = train_test_split(
    files,
    labels,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=labels
)

print('Train trials:', len(train_files))
print('Test trials :', len(test_files))
print('Train label counts:', dict(pd.Series(train_labels).value_counts().sort_index()))
print('Test label counts :', dict(pd.Series(test_labels).value_counts().sort_index()))


## 4. 전처리 함수

- 60 Hz Notch Filter (Q=30)
- 20~499 Hz 4차 Butterworth Band-pass Filter
- 300 ms sliding window
- 50% overlap (hop=150 samples)
- Window 단위 Min-Max normalization
- Morlet CWT, scale 1~32
- 2개 채널 CWT + 두 채널 평균맵 = 최종 입력 `(3, 32, 300)`


In [ ]:
def filter_semg(x, fs=FS):
    # 60 Hz notch
    b_notch, a_notch = iirnotch(60, Q=30, fs=fs)
    x = filtfilt(b_notch, a_notch, x, axis=0)

    # 20~499 Hz band-pass
    b_band, a_band = butter(4, [20, 499], btype='bandpass', fs=fs)
    x = filtfilt(b_band, a_band, x, axis=0)
    return x

def make_windows(x, win=WIN, hop=HOP):
    return [x[s:s+win] for s in range(0, len(x)-win+1, hop)]

def minmax_window(w):
    mn, mx = w.min(), w.max()
    return (w - mn) / (mx - mn + 1e-12)

def to_cwt(w):
    maps = []
    for ch in range(2):
        coef, _ = pywt.cwt(w[:, ch], SCALES, 'morl')
        maps.append(np.abs(coef))
    maps = np.stack(maps, axis=0)
    mean_map = maps.mean(axis=0, keepdims=True)
    return np.concatenate([maps, mean_map], axis=0).astype(np.float32)

label_map = {'A':0, 'B':1, 'C':2, 'D':3, 'E':4}
class_names = ['A','B','C','D','E']

def build_dataset(file_list):
    X, y = [], []
    for f in file_list:
        x = np.loadtxt(f, delimiter=',', skiprows=1)
        if x.shape[0] < x.shape[1]:
            x = x.T
        x = filter_semg(x)
        label = label_map[os.path.basename(os.path.dirname(f))]
        for w in make_windows(x):
            X.append(to_cwt(minmax_window(w)))
            y.append(label)
    return np.stack(X), np.asarray(y, dtype=np.int64)


## 5. 전처리 데이터 생성

각 3초 trial에서 300 ms window, 50% overlap을 적용하면 19개 window가 생성된다.  
따라서 Train 200 trials → 3800 windows, Test 50 trials → 950 windows가 된다.


In [ ]:
# CWT 변환은 시간이 걸린다.
# 이미 저장된 Xtr/Xte가 있다면 이 셀 대신 np.load로 불러와도 된다.

Xtr, ytr = build_dataset(train_files)
Xte, yte = build_dataset(test_files)

print('Xtr:', Xtr.shape, 'ytr:', ytr.shape)
print('Xte:', Xte.shape, 'yte:', yte.shape)

assert Xtr.shape == (3800, 3, 32, 300)
assert Xte.shape == (950, 3, 32, 300)


## 6. DataLoader 및 모델 정의


In [ ]:
def make_loaders(X_train, y_train, X_test, y_test, batch_size=BATCH_SIZE):
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
        batch_size=batch_size, shuffle=True
    )
    test_loader = DataLoader(
        TensorDataset(torch.tensor(X_test), torch.tensor(y_test)),
        batch_size=batch_size, shuffle=False
    )
    return train_loader, test_loader

def make_2dcnn():
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(64, 5)
    ).to(device)

def make_resnet18():
    m = resnet18(weights=None)
    m.fc = nn.Linear(512, 5)
    return m.to(device)

def make_densenet161():
    m = densenet161(weights=None)
    m.classifier = nn.Linear(2208, 5)
    return m.to(device)

for name, model in [
    ('2D CNN', make_2dcnn()),
    ('ResNet18', make_resnet18()),
    ('DenseNet161', make_densenet161())
]:
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{name:12s}: {params:,} params')
    del model
torch.cuda.empty_cache()


## 7. 학습 및 평가 함수

모든 모델은 45 epochs, batch size 16, Adam(lr=0.001), CrossEntropyLoss, pretrained weight 미사용 조건으로 학습한다.


In [ ]:
def train_and_evaluate(model, train_loader, test_loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.time()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device, dtype=torch.float32)
            yb = yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:2d}/{epochs} | loss={total_loss/len(train_loader):.4f}')

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    train_time = time.time() - start

    model.eval()
    preds, trues, probs = [], [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            out = model(xb.to(device, dtype=torch.float32))
            prob = torch.softmax(out, dim=1)
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())
            probs.extend(prob.cpu().numpy())

    acc = accuracy_score(trues, preds)
    precision = precision_score(trues, preds, average='macro')
    recall = recall_score(trues, preds, average='macro')
    f1 = f1_score(trues, preds, average='macro')
    cm = confusion_matrix(trues, preds)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'Train Time': train_time,
        'Parameters': params,
        'CM': cm,
        'preds': np.asarray(preds),
        'trues': np.asarray(trues),
        'probs': np.asarray(probs),
    }


## 8. 3회 반복 실험

아래 셀을 실행하면 동일한 Train/Test split에서 각 모델을 3회씩 새로 학습한다. 시간이 오래 걸리므로 제출본 확인만 할 때는 `RUN_FULL_EXPERIMENT=False`로 둔다.


In [ ]:
RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:
    train_loader, test_loader = make_loaders(Xtr, ytr, Xte, yte)
    factories = {
        '2D CNN': make_2dcnn,
        'ResNet18': make_resnet18,
        'DenseNet161': make_densenet161
    }

    all_runs = {}
    for name, factory in factories.items():
        all_runs[name] = []
        for run in range(1, 4):
            print(f'\n===== {name} | Run {run} =====')
            model = factory()
            result = train_and_evaluate(model, train_loader, test_loader)
            all_runs[name].append(result)
            print({k: result[k] for k in ['Accuracy','Precision','Recall','F1','Train Time','Parameters']})
            del model
            torch.cuda.empty_cache()


## 9. 기록된 최종 3회 평균 성능

아래 값은 실제 완료한 3회 실험의 평균이다.


In [ ]:
summary = pd.DataFrame([
    ['2D CNN',     63.23, 65.29, 63.23, 62.95],
    ['ResNet18',   87.79, 88.19, 87.79, 87.71],
    ['DenseNet161',87.44, 88.17, 87.45, 87.34],
], columns=['Model','Accuracy (%)','Precision (%)','Recall (%)','F1-score (%)'])

summary


## 10. Confusion Matrix

대표 실행의 confusion matrix를 사용한다.


In [ ]:
cm_2dcnn = np.array([
    [183, 3, 0, 1, 3],
    [1, 150, 21, 15, 3],
    [1, 39, 116, 25, 9],
    [0, 31, 29, 109, 21],
    [0, 87, 13, 28, 62]
])

cm_resnet18 = np.array([
    [188, 1, 1, 0, 0],
    [3, 168, 9, 3, 7],
    [0, 2, 176, 6, 6],
    [2, 8, 17, 161, 2],
    [1, 14, 7, 7, 161]
])

cm_densenet161 = np.array([
    [190, 0, 0, 0, 0],
    [5, 146, 6, 9, 24],
    [0, 0, 167, 19, 4],
    [0, 3, 3, 182, 2],
    [1, 12, 3, 10, 164]
])

os.makedirs('results', exist_ok=True)

def save_cm(cm, title, filename):
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(6,5))
    disp.plot(ax=ax, values_format='d', colorbar=True)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(f'results/{filename}', dpi=150)
    plt.show()
    plt.close()

save_cm(cm_2dcnn, '2D CNN Confusion Matrix', 'confusion_2dcnn.png')
save_cm(cm_resnet18, 'ResNet18 Confusion Matrix', 'confusion_resnet18.png')
save_cm(cm_densenet161, 'DenseNet161 Confusion Matrix', 'confusion_densenet.png')


### 대표 오분류

- 2D CNN: E → B, 87개
- ResNet18: D → C, 17개
- DenseNet161: B → E, 24개

DenseNet161의 B→E 오류 24건은 10개 B trial에 분산되어 있었고, 16건(66.7%)이 Rotation 구간에서 발생했다. 오분류 window에서 E 클래스 평균 예측 확률은 약 0.760이었다.


## 11. DenseNet161 5-Fold Cross Validation

Fold는 **trial 파일 단위**로 나눈다. 전체 실행은 오래 걸리므로 `RUN_CV=True`로 바꿨을 때만 수행한다.


In [ ]:
RUN_CV = False

def train_fold(model, X_train, y_train, X_valid, y_valid):
    train_loader, valid_loader = make_loaders(
        X_train, y_train, X_valid, y_valid
    )
    return train_and_evaluate(model, train_loader, valid_loader)['Accuracy']

if RUN_CV:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(files, labels), start=1):
        print(f'\n===== Fold {fold} =====')
        X_fold_tr, y_fold_tr = build_dataset([files[i] for i in tr_idx])
        X_fold_va, y_fold_va = build_dataset([files[i] for i in va_idx])

        model = make_densenet161()
        acc = train_fold(model, X_fold_tr, y_fold_tr, X_fold_va, y_fold_va)
        cv_scores.append(acc)
        print(f'Fold {fold}: {acc*100:.2f}%')

        del model, X_fold_tr, y_fold_tr, X_fold_va, y_fold_va
        torch.cuda.empty_cache()

    print(f'Mean: {np.mean(cv_scores)*100:.2f} ± {np.std(cv_scores)*100:.2f}%')


### 기록된 5-Fold 결과


In [ ]:
fold_scores = np.array([87.47, 89.05, 87.16, 88.32, 89.37])
fold_table = pd.DataFrame({
    'Fold': [f'Fold {i}' for i in range(1,6)],
    'Accuracy (%)': fold_scores
})

display(fold_table)
print(f'Mean Accuracy: {fold_scores.mean():.2f} ± {fold_scores.std():.2f}%')


## 12. 계산 비용 비교

학습시간은 실제 기록값을 사용한다.


In [ ]:
cost_table = pd.DataFrame([
    ['2D CNN',      63.23, 62.95,   56.8,   19717],
    ['ResNet18',    87.79, 87.71,  236.3, 11179077],
    ['DenseNet161', 87.44, 87.34, 1247.2, 26483045],
], columns=['Model','Accuracy (%)','F1-score (%)','Training Time (s)','Parameters'])

cost_table


## 13. 최종 요약

- 3회 평균 기준 ResNet18: Accuracy 87.79%, F1 87.71%
- DenseNet161: Accuracy 87.44%, F1 87.34%
- 2D CNN: Accuracy 63.23%, F1 62.95%
- DenseNet161 5-Fold: **88.27 ± 0.86%**
- DenseNet161 대표 실행의 최대 오분류: **B → E 24건**
- 대표 실행에서 사용자 A는 DenseNet161이 **190/190**으로 모두 정확히 분류

결과는 실행 환경과 모델 초기화 등에 따라 일부 달라질 수 있다.
